In [2]:
import geopandas as gpd
import polars as pl
import polars_h3 as plh3
import yaml
from datetime import datetime


In [3]:
PARTITION = 0
DIR = "/mnt/common-hdd/group-dirs/tkom-2026-project/stops/"
OUTPUT = "../output/"

In [4]:
data = pl.read_parquet(f"{DIR}/{PARTITION:02x}.parquet")
data.head(3)

timestamp,device_id,device_aid_type,latitude,longitude,stop_label
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,2
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,2
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,-1


In [ ]:
print(data.head(3).to_pandas().to_markdown(index=False))

| timestamp                 |   device_id | device_aid_type   |   latitude |   longitude |   stop_label |
|:--------------------------|------------:|:------------------|-----------:|------------:|-------------:|
| 2024-10-02 02:37:09+02:00 |           1 | AAID              |    47.4799 |     19.0742 |            2 |
| 2024-10-02 03:17:49+02:00 |           1 | AAID              |    47.48   |     19.0742 |            2 |
| 2024-10-02 03:26:50+02:00 |           1 | AAID              |    47.4768 |     19.0752 |           -1 |


In [24]:
data = data.with_columns(
    plh3.int_to_str(
        plh3.latlng_to_cell(pl.col("latitude"), pl.col("longitude"), 8)
    ).alias("h3_8"),
    plh3.int_to_str(
        plh3.latlng_to_cell(pl.col("latitude"), pl.col("longitude"), 9)
    ).alias("h3_9"),
    plh3.int_to_str(
        plh3.latlng_to_cell(pl.col("latitude"), pl.col("longitude"), 10)
    ).alias("h3_10"),
    pl.col("timestamp").dt.date().alias("day"),
    pl.col("timestamp").dt.time().alias("time"),
    pl.col("timestamp").dt.weekday().alias("dow"),
)

In [25]:
data.head(3)

timestamp,device_id,device_aid_type,latitude,longitude,stop_label,h3_8,h3_9,h3_10,day,time,dow
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64,str,str,str,date,time,i8
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,2,"""881e0378c1fffff""","""891e0378c1bffff""","""8a1e0378c1affff""",2024-10-02,02:37:09,3
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,2,"""881e0378c1fffff""","""891e0378c1bffff""","""8a1e0378c1affff""",2024-10-02,03:17:49,3
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,-1,"""881e0378c1fffff""","""891e0378c0bffff""","""8a1e0378c087fff""",2024-10-02,03:26:50,3


In [26]:
h3_8_count = data.group_by(["h3_8"]).len().rename({"h3_8": "h3", "len": "count"})
h3_9_count = data.group_by(["h3_9"]).len().rename({"h3_9": "h3", "len": "count"})
h3_10_count = data.group_by(["h3_10"]).len().rename({"h3_10": "h3", "len": "count"})

In [27]:
with open("06_home_detection_config.yaml") as fp:
    config = yaml.safe_load(fp)

In [17]:
def detect_home_work_pl(
    data: pl.DataFrame,
    config: dict,
    spatial_id: str = "h3",
    device_id: str = "device_id",
) -> pl.DataFrame:
    home_hours_end = datetime.strptime(config["home_hours"]["end"], "%H:%M").time()
    home_hours_begin = datetime.strptime(config["home_hours"]["begin"], "%H:%M").time()
    work_hours_end = datetime.strptime(config["work_hours"]["end"], "%H:%M").time()
    work_hours_begin = datetime.strptime(config["work_hours"]["begin"], "%H:%M").time()

    home_stay_count = (
        data.filter(
            (
                (
                    (pl.col("time") < home_hours_end)
                    | (pl.col("time") > home_hours_begin)
                )
                & (pl.col("dow") < 6)
            )
            | ((pl.col("dow") >= 6) | (pl.col("day").is_in(config["weekday_holidays"])))
        )
        .group_by([device_id, spatial_id])
        .agg(pl.col("day").count())
        .rename({"day": "count"})
        .sort([device_id, "count"], descending=[False, True])
        .unique(subset=device_id, keep="first")
    )
    work_stay_count = (
        data.filter(
            ((pl.col("time") > work_hours_begin) & (pl.col("time") < work_hours_end))
            & ((pl.col("dow") < 6) | (pl.col("day").is_in(config["weekend_workdays"])))
            & (~pl.col("day").is_in(config["weekday_holidays"]))
        )
        .group_by([device_id, spatial_id])
        .agg(pl.col("day").count())
        .rename({"day": "count"})
        .sort([device_id, "count"], descending=[False, True])
        .unique(subset=device_id, keep="first")
    )
    return home_stay_count.join(
        work_stay_count,
        on=device_id,
        how="full",
        suffix="_work",
        coalesce=True,
    ).rename({spatial_id: f"{spatial_id}_home", "count": "count_home"})

In [ ]:
hw8 = detect_home_work_pl(data, config, spatial_id="h3_8", device_id="device_id")
hw8.write_csv(f"{OUTPUT}/home_work_{PARTITION:02x}_h3_8.csv")
hw9 = detect_home_work_pl(data, config, spatial_id="h3_9", device_id="device_id")
hw9.write_csv(f"{OUTPUT}/home_work_{PARTITION:02x}_h3_9.csv")
hw10 = detect_home_work_pl(data, config, spatial_id="h3_10", device_id="device_id")
hw10.write_csv(f"{OUTPUT}/home_work_{PARTITION:02x}_h3_10.csv")

Detect home in settlements

In [ ]:
settlements = gpd.read_file("../data/osm/hungary/settlements.geojson").set_crs(4326)
settlements = settlements[settlements["name"] != "Budapest"].copy()

In [30]:
def map_to_settlements(
    data: pl.DataFrame, settlements: gpd.GeoDataFrame
) -> pl.DataFrame:
    gdf = data.to_pandas()  # 9s
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf["longitude"], gdf["latitude"]),
        crs=4326,
    )
    # +23s
    gdf = gdf.sjoin(settlements).drop(["index_right", "geometry"], axis=1)
    # + 90s
    gdf = pl.from_pandas(gdf)
    # +13s
    return gdf

In [ ]:
data_s = map_to_settlements(data, settlements)
# runs 2m 18s for p00

In [32]:
data_s.head(3)

timestamp,device_id,device_aid_type,latitude,longitude,stop_label,h3_8,h3_9,h3_10,day,time,dow,osm_id,name
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64,str,str,str,datetime[ms],time,i8,i32,str
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,2,"""881e0378c1fffff""","""891e0378c1bffff""","""8a1e0378c1affff""",2024-10-02 00:00:00,02:37:09,3,1552462,"""IX. kerület"""
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,2,"""881e0378c1fffff""","""891e0378c1bffff""","""8a1e0378c1affff""",2024-10-02 00:00:00,03:17:49,3,1552462,"""IX. kerület"""
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,-1,"""881e0378c1fffff""","""891e0378c0bffff""","""8a1e0378c087fff""",2024-10-02 00:00:00,03:26:50,3,1552462,"""IX. kerület"""


In [ ]:
hws = detect_home_work_pl(data_s, config, spatial_id="name", device_id="device_id")
hws.write_csv(f"{OUTPUT}/home_work_{PARTITION:02x}_settlement.csv")